In [9]:
%pip install --upgrade pip
%pip install pandas 

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import pandas as pd

RAW_STOCK_DIR = Path(r"E:/stock-price-prediction/data/raw/stock")

tickers = ["VCB", "FPT" , "VIC","HPG", "VNM"]  # thêm MWG chỉ khi có file MWG.csv
trading_days_by_ticker = {}

def find_date_col(df):
    candidates = ["time", "trading_date", "date", "tradingDate"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Không tìm thấy cột ngày. Các cột hiện có: {df.columns.tolist()}")

for ticker in tickers:
    file_path = RAW_STOCK_DIR / f"{ticker}.csv"

    if not file_path.exists():
        print(f"[WARNING] Không tìm thấy file: {file_path}")
        continue

    df = pd.read_csv(file_path)

    date_col = find_date_col(df)
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    dates = set(df[date_col].dropna().dt.date)
    trading_days_by_ticker[ticker] = dates

if not trading_days_by_ticker:
    raise ValueError("Không load được dữ liệu của mã nào.")

common_trading_days = set.intersection(*trading_days_by_ticker.values())

print(f"Loaded tickers: {list(trading_days_by_ticker.keys())}")
print(f"Number of common trading days: {len(common_trading_days)}")

if common_trading_days:
    print(f"Frm: {min(common_trading_days)}")
    print(f"To:   {max(common_trading_days)}")
else:
    print("Không có ngày giao dịch chung giữa tất cả các mã.")

summary = pd.DataFrame({
    "ticker": list(trading_days_by_ticker.keys()),
    "total_trading_days": [
        len(trading_days_by_ticker[t]) for t in trading_days_by_ticker.keys()
    ],
    "common_trading_days": len(common_trading_days),
    "missing_vs_common": [
        len(common_trading_days - trading_days_by_ticker[t])
        for t in trading_days_by_ticker.keys()
    ],
})

print(summary)

Loaded tickers: ['VCB', 'FPT', 'VIC', 'HPG', 'VNM']
Number of common trading days: 4125
From: 2009-06-30
To:   2025-12-31
  ticker  total_trading_days  common_trading_days  missing_vs_common
0    VCB                4125                 4125                  0
1    FPT                4750                 4125                  0
2    VIC                4562                 4125                  0
3    HPG                4521                 4125                  0
4    VNM                4976                 4125                  0
